# Lab 3: Linear Spaces, Subspaces, Linear Transformations, Quotients, and Tensors

**MATH 5110 – Applied Linear Algebra and Matrix Analysis**  
**Independent-study lab with solutions**

This lab supports Chapter 3. It is written for independent study: each main problem is followed by a worked solution and a similar practice question.

Main themes:

- vector spaces and subspaces;
- spans and membership;
- direct sums;
- linear transformations, kernels, and images;
- quotient spaces;
- tensor products and Kronecker products.

## 0. Setup

Run this cell first. We use `sympy` for exact symbolic linear algebra.

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

sp.init_printing()

def rref_with_pivots(M):
    M = sp.Matrix(M)
    R, pivots = M.rref()
    return R, pivots

def display_rref(M):
    R, pivots = rref_with_pivots(M)
    print("Matrix:")
    display(sp.Matrix(M))
    print("RREF:")
    display(R)
    print("Pivot columns (0-indexed):", pivots)
    print("Pivot columns (1-indexed):", tuple(i+1 for i in pivots))
    return R, pivots

def column_space_from_pivots(A, pivots):
    A = sp.Matrix(A)
    return [A[:, j] for j in pivots]

## 1. Subspaces: zero vector and closure

A subset $U$ of a vector space $V$ is a subspace if:

1. $0\in U$;
2. $u,v\in U$ implies $u+v\in U$;
3. $u\in U$ and $c\in\mathbb F$ implies $cu\in U$.

A very common source of subspaces is a homogeneous linear equation.

### Problem 1

Determine whether each set is a subspace of the indicated vector space.

1. $U=\{(x,y)\in\mathbb R^2:2x-y=0\}$.
2. $A=\{(x,y)\in\mathbb R^2:2x-y=1\}$.
3. $C=\{(x,y)\in\mathbb R^2:x\ge 0, y\ge 0\}$.

### Solution 1

1. $U$ is a subspace. It is the kernel of the linear functional
   $$T(x,y)=2x-y.$$
   Kernels of linear transformations are subspaces.

2. $A$ is not a subspace because $(0,0)\notin A$. In fact,
   $$2(0)-0=0\ne 1.$$
   This is an affine line, not a linear subspace.

3. $C$ is not a subspace. It is closed under addition and nonnegative scalar multiplication, but not under arbitrary scalar multiplication. For example,
   $$(1,1)\in C,\qquad -1(1,1)=(-1,-1)\notin C.$$

### Similar practice 1

Determine whether
$$H=\{(x,y,z)\in\mathbb R^3:x+2y-z=0\}$$
is a subspace.

**Answer.** Yes. It is the kernel of $T(x,y,z)=x+2y-z$.

## 2. Polynomials as vectors and span membership

A polynomial in $\mathbb R_3[t]$ can be represented by its coordinate vector in the basis
$$ (1,t,t^2,t^3). $$

For example,
$$2-3t+5t^3 \longleftrightarrow \begin{bmatrix}2\\-3\\0\\5\end{bmatrix}.$$

Consider
$$p_1(t)=1+t,\qquad p_2(t)=t+t^2,\qquad p_3(t)=t^2+t^3.$$

In [ ]:
p1 = sp.Matrix([1, 1, 0, 0])
p2 = sp.Matrix([0, 1, 1, 0])
p3 = sp.Matrix([0, 0, 1, 1])
P = sp.Matrix.hstack(p1, p2, p3)
display(P)
R, pivots = display_rref(P)

### Solution 2.1

The pivot columns are all three columns, so $p_1,p_2,p_3$ are linearly independent.

The span is a 3-dimensional subspace of $\mathbb R_3[t]$.

### Problem 2.2

Let
$$q(t)=\alpha+\beta t+\gamma t^2+\delta t^3.$$

Find the condition on $(\alpha,\beta,\gamma,\delta)$ for $q(t)$ to belong to
$$W=\operatorname{span}(p_1,p_2,p_3).$$

In [ ]:
alpha, beta, gamma, delta = sp.symbols('alpha beta gamma delta')
a, b, c = sp.symbols('a b c')
q = sp.Matrix([alpha, beta, gamma, delta])
equations = sp.Eq(P*sp.Matrix([a,b,c]), q)
display(P*sp.Matrix([a,b,c]))
display(q)

### Solution 2.2

The equation
$$a p_1+b p_2+c p_3=q$$
gives
$$
a=\alpha,\qquad a+b=\beta,\qquad b+c=\gamma,\qquad c=\delta.
$$

Eliminate $a,b,c$:

$$
a=\alpha,
\qquad b=\beta-\alpha,
\qquad c=\delta.
$$

Therefore
$$
\gamma=b+c=\beta-\alpha+\delta.
$$

Thus
$$
q(t)\in W\quad\Longleftrightarrow\quad \gamma=\beta-\alpha+\delta.
$$

In [ ]:
def in_polynomial_span(qvec):
    try:
        coeffs = P.gauss_jordan_solve(sp.Matrix(qvec))[0]
        return True, coeffs
    except ValueError:
        return False, None

examples = {
    "q1 = 1 + 2t + 2t^2 + t^3": [1, 2, 2, 1],
    "q2 = 1 + 2t + 3t^2 + t^3": [1, 2, 3, 1],
}

for name, qvec in examples.items():
    ok, coeffs = in_polynomial_span(qvec)
    print(name)
    print("belongs to W?", ok)
    if ok:
        print("coordinates in p1,p2,p3:")
        display(coeffs)
    print()

### Similar practice 2

Decide whether
$$r(t)=2+5t+4t^2+t^3$$
belongs to $W$.

**Answer.** Here $\alpha=2,\beta=5,\delta=1$, so $\beta-\alpha+\delta=5-2+1=4=\gamma$. Hence $r\in W$.

## 3. A plane as a span

Let
$$u_1=(1,0,2),\qquad u_2=(0,1,-1).$$

Then
$$
s u_1+t u_2=(s,t,2s-t).
$$

In [ ]:
u1 = sp.Matrix([1, 0, 2])
u2 = sp.Matrix([0, 1, -1])
U = sp.Matrix.hstack(u1, u2)
display(U)

s, t = sp.symbols('s t')
display(s*u1 + t*u2)

### Solution 3

Since a general vector in the span has the form
$$
(s,t,2s-t),
$$
we get the plane
$$
W=\{(x,y,z)\in\mathbb R^3:z=2x-y\}.
$$

Equivalently,
$$2x-y-z=0.$$

In [ ]:
def solve_membership_R3(w):
    w = sp.Matrix(w)
    try:
        sol = U.gauss_jordan_solve(w)[0]
        return True, sol
    except ValueError:
        return False, None

for w in [sp.Matrix([3,4,2]), sp.Matrix([3,4,1]), sp.Matrix([2,-1,5])]:
    ok, sol = solve_membership_R3(w)
    print("w =", list(w))
    print("belongs to span(u1,u2)?", ok)
    if ok:
        print("coefficients s,t:")
        display(sol)
    print()

### Similar practice 3

Decide whether $w=(2,-1,6)$ belongs to $\operatorname{span}(u_1,u_2)$.

**Answer.** No. The first two coordinates force $s=2$ and $t=-1$, so the third coordinate would be $2s-t=5$, not $6$.

## 4. Direct sums

Let
$$
U=\{(x,0,0):x\in\mathbb R\},\qquad
W=\{(0,y,z):y,z\in\mathbb R\}.
$$

### Solution 4

Every vector $(a,b,c)\in\mathbb R^3$ can be written as
$$
(a,b,c)=(a,0,0)+(0,b,c),
$$
where $(a,0,0)\in U$ and $(0,b,c)\in W$.

Also,
$$
U\cap W=\{(0,0,0)\}.
$$

Therefore
$$
\mathbb R^3=U\oplus W.
$$

### Similar practice 4

Let
$$
U=\{(x,y,0):x,y\in\mathbb R\},\qquad
W=\{(0,y,z):y,z\in\mathbb R\}.
$$

Is $\mathbb R^3=U\oplus W$?

**Answer.** No. Although $U+W=\mathbb R^3$, the intersection is
$$U\cap W=\{(0,y,0):y\in\mathbb R\},$$
which is not $\{0\}$.

## 5. Kernel and image from a matrix

Let $T:\mathbb R^4\to\mathbb R^3$ be defined by $T(x)=Ax$, where

$$
A=\begin{bmatrix}
0&0&2&8\\
1&5&2&-5\\
2&10&6&-2
\end{bmatrix}.
$$

In [ ]:
A = sp.Matrix([[0,0,2,8],
               [1,5,2,-5],
               [2,10,6,-2]])
R, pivots = display_rref(A)

In [ ]:
image_basis = column_space_from_pivots(A, pivots)
kernel_basis = A.nullspace()

print("Image basis from original pivot columns:")
for v in image_basis:
    display(v)

print("Kernel basis:")
for v in kernel_basis:
    display(v)

print("rank =", A.rank())
print("nullity =", len(kernel_basis))
print("rank + nullity =", A.rank() + len(kernel_basis))

### Solution 5

The pivot columns are columns 1 and 3 of the original matrix. Therefore

$$
\operatorname{im}(T)=\operatorname{span}\left\{
\begin{bmatrix}0\\1\\2\end{bmatrix},
\begin{bmatrix}2\\2\\6\end{bmatrix}
\right\}.
$$

The RREF equations are

$$
x_1+5x_2-13x_4=0,
\qquad
x_3+4x_4=0.
$$

Let $x_2=s$ and $x_4=t$. Then

$$
x=s\begin{bmatrix}-5\\1\\0\\0\end{bmatrix}
+t\begin{bmatrix}13\\0\\-4\\1\end{bmatrix}.
$$

Hence

$$
\ker(T)=\operatorname{span}\left\{
\begin{bmatrix}-5\\1\\0\\0\end{bmatrix},
\begin{bmatrix}13\\0\\-4\\1\end{bmatrix}
\right\}.
$$

### Similar practice 5

Let
$$
B=\begin{bmatrix}
1&0&0&2\\
0&1&0&-1\\
0&0&1&3
\end{bmatrix}.
$$

Find $\ker(B)$.

In [ ]:
B = sp.Matrix([[1,0,0,2],
               [0,1,0,-1],
               [0,0,1,3]])
display_rref(B)
print("Kernel basis:")
for v in B.nullspace():
    display(v)

**Answer.** The equations are
$$x_1+2x_4=0,\qquad x_2-x_4=0,\qquad x_3+3x_4=0.$$
Let $x_4=t$. Then
$$
x=t\begin{bmatrix}-2\\1\\-3\\1\end{bmatrix}.
$$
Thus
$$
\ker(B)=\operatorname{span}\left\{\begin{bmatrix}-2\\1\\-3\\1\end{bmatrix}\right\}.
$$

## 6. Quotient spaces

If $N$ is a subspace of a finite-dimensional vector space $V$, then
$$
\dim(V/N)=\dim(V)-\dim(N).
$$

Intuitively, $V/N$ treats all vectors differing by an element of $N$ as equivalent.

### Problem 6

Let $N$ be a 2-dimensional subspace of $\mathbb R^4$. Find $\dim(\mathbb R^4/N)$.

### Solution 6

$$
\dim(\mathbb R^4/N)=4-2=2.
$$

### Similar practice 6

Let $N=\operatorname{span}\{(1,0,0),(0,1,0)\}\subseteq\mathbb R^3$. Find $\dim(\mathbb R^3/N)$.

**Answer.** $3-2=1$.

## 7. Tensor products and Kronecker products

For matrices, the Kronecker product gives a concrete computational model related to tensor products.

If
$$
A=\begin{bmatrix}a&b\\c&d\end{bmatrix},
$$
then
$$
A\otimes B=
\begin{bmatrix}
aB&bB\\
cB&dB
\end{bmatrix}.
$$

In [ ]:
A2 = sp.Matrix([[1,2],[3,4]])
B2 = sp.Matrix([[0,5],[6,7]])
kron = sp.kronecker_product(A2, B2)
display(A2)
display(B2)
display(kron)

### Similar practice 7

Compute $A\otimes B$ for
$$
A=\begin{bmatrix}0&1\\1&0\end{bmatrix},
\qquad
B=\begin{bmatrix}1&0\\0&-1\end{bmatrix}.
$$

In [ ]:
A_practice = sp.Matrix([[0,1],[1,0]])
B_practice = sp.Matrix([[1,0],[0,-1]])
display(sp.kronecker_product(A_practice, B_practice))

## 8. Final independent-study questions

1. Give one example of a subspace and one example of a non-subspace in $\mathbb R^2$.
2. Explain why every kernel of a linear transformation is a subspace.
3. Explain why the image of a linear transformation is a subspace.
4. State the rank-nullity theorem.
5. In your own words, explain what a quotient space $V/N$ does.
6. Ask an AI tool to explain quotient spaces. Critique whether the answer correctly uses cosets and the dimension formula.

### Brief answer key

1. Example subspace: $\{(x,y):x-y=0\}$. Example non-subspace: $\{(x,y):x-y=1\}$.
2. If $T(u)=0$ and $T(v)=0$, then $T(u+v)=0$ and $T(cu)=0$.
3. If $y_1=T(v_1)$ and $y_2=T(v_2)$, then $y_1+y_2=T(v_1+v_2)$ and $cy_1=T(cv_1)$.
4. If $T:V\to W$ and $V$ is finite-dimensional, then $\dim V=\dim\ker(T)+\dim\operatorname{im}(T)$.
5. It identifies vectors that differ by elements of $N$; equivalently, it collapses $N$ to zero.
6. A good answer should mention cosets $v+N$, the natural projection $V\to V/N$, and $\dim(V/N)=\dim V-\dim N$ in finite dimensions.